<a href="https://colab.research.google.com/github/minhthieukekw-cmd/thpthamrong/blob/main/THPT_Ha%CC%80m_Ro%CC%82%CC%80ng.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils import resample
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# 1. XỬ LÝ DỮ LIỆU GỐC (HÀM RỒNG DATA)
# ==========================================
df_hk1 = pd.read_excel("Goc  Thi Hoc ky1-K112-24-25 - Up.xlsx", sheet_name='goc')
df_thithu = pd.read_excel("Thi thu THPT Quoc gia lan1_So_2024-25.GOC.xlsx", sheet_name='goc')

# Làm sạch HK1
hk1_clean = df_hk1.iloc[1:].copy()
hk1_clean.columns = df_hk1.iloc[0].tolist()
features = ['toan', 'van', 'anh', 'li', 'hoa', 'su']
for col in features:
    hk1_clean[col] = pd.to_numeric(hk1_clean[col], errors='coerce').fillna(0)

# Làm sạch Thi thử (Nguyện vọng thực tế)
thithu_clean = df_thithu.iloc[3:].copy()
thithu_scores = thithu_clean.iloc[:, [1, 23, 24, 25, 26, 27, 28, 29, 30]].copy()
thithu_scores.columns = ['SBD', 'T1_Toan', 'T1_Van', 'T1_Anh', 'T1_Ly', 'T1_Hoa', 'T1_Sinh', 'T1_Su', 'T1_Dia']

# Khớp nối SBD
hk1_clean['SBD'] = hk1_clean['SBD'].astype(str).str.strip()
thithu_scores['SBD'] = thithu_scores['SBD'].astype(str).str.strip()
df_merged = pd.merge(hk1_clean[['SBD'] + features], thithu_scores, on='SBD', how='inner')

# Gán nhãn khối thi (Strict Logic)
def get_label_final(row):
    su = pd.to_numeric(row['T1_Su'], errors='coerce')
    dia = pd.to_numeric(row['T1_Dia'], errors='coerce')
    ly = pd.to_numeric(row['T1_Ly'], errors='coerce')
    hoa = pd.to_numeric(row['T1_Hoa'], errors='coerce')
    anh = pd.to_numeric(row['T1_Anh'], errors='coerce')
    if su > 0 and dia > 0: return 'C00'
    if ly > 0 and hoa > 0: return 'A00'
    if ly > 0 and anh > 0: return 'A01'
    if anh > 0: return 'D01'
    return 'Khác'

df_merged['Target'] = df_merged.apply(get_label_final, axis=1)
df_ml = df_merged[df_merged['Target'] != 'Khác'].copy()

# ==========================================
# 2. HUẤN LUYỆN AI & CÂN BẰNG DỮ LIỆU
# ==========================================
max_size = df_ml['Target'].value_counts().max()
df_balanced = pd.concat([resample(df_ml[df_ml.Target == c], replace=True, n_samples=max_size, random_state=42)
                         for c in df_ml['Target'].unique()])

rf_final = RandomForestClassifier(n_estimators=300, max_depth=15, class_weight='balanced', random_state=42)
rf_final.fit(df_balanced[features], df_balanced['Target'])

# ==========================================
# 3. HIỂN THỊ BÁO CÁO CHI TIẾT 100%
# ==========================================
print("\n" + "="*60)
print("BÁO CÁO PHÂN TÍCH DỮ LIỆU AI - TRƯỜNG THPT HÀM RỒNG")
print("="*60)

# Bảng số liệu
stats = df_ml['Target'].value_counts().reset_index()
stats.columns = ['Khối thi', 'Số lượng HS']
print("\n[BẢNG 1] THỐNG KÊ XU HƯỚNG CHỌN KHỐI THỰC TẾ:")
print(stats.to_string(index=False))

# Biểu đồ quan trọng
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.barplot(data=stats, x='Khối thi', y='Số lượng HS', palette='viridis')
plt.title('Phân bổ khối thi thực tế')

plt.subplot(1, 2, 2)
importances = pd.Series(rf_final.feature_importances_, index=['Toán', 'Văn', 'Anh', 'Lý', 'Hóa', 'Sử'])
importances.sort_values().plot(kind='barh', color='salmon')
plt.title('Mức độ ảnh hưởng của điểm số đến dự báo')
plt.tight_layout()
plt.show()

# ==========================================
# 4. HÀM AI CONSULTANT CHUYÊN SÂU
# ==========================================
def ai_consultant_pro(toan, van, anh, ly, hoa, su):
    # 1. Dự đoán bằng AI
    input_df = pd.DataFrame([[toan, van, anh, ly, hoa, su]], columns=features)
    pred_block = rf_final.predict(input_df)[0]
    probs = rf_final.predict_proba(input_df)[0]
    conf = max(probs) * 100

    # 2. Logic phân tích thế mạnh (Giải thích lý do)
    reasons = []
    if su >= 8.0 and su > anh:
        reasons.append(f"Điểm môn Lịch sử ({su}) vượt trội, phù hợp với tư duy học thuật khối C.")
    if ly >= 8.0 and hoa >= 8.0:
        reasons.append(f"Cặp điểm Lý ({ly}) - Hóa ({hoa}) cho thấy năng lực tư duy logic mạnh.")
    if anh >= 8.0 and toan >= 8.0:
        reasons.append(f"Sự kết hợp giữa Toán ({toan}) và Ngoại ngữ ({anh}) mở ra cơ hội lớn ở các khối A01, D01.")

    # 3. Lời khuyên nghề nghiệp
    career_map = {
        'D01': 'Kinh tế, Ngôn ngữ, Sư phạm, Marketing, Luật.',
        'A01': 'Công nghệ thông tin, Điện tử viễn thông, Quản trị kinh doanh.',
        'A00': 'Kỹ thuật, Xây dựng, Bách khoa, Công nghệ sản xuất.',
        'C00': 'Báo chí, Truyền thông, Quân đội, Công an, Luật.',
        'B00': 'Y khoa, Dược học, Công nghệ sinh học, Môi trường.'
    }

    print("\n" + "*"*20 + " KẾT QUẢ TƯ VẤN TỪ HỆ THỐNG AI " + "*"*20)
    print(f"▶ KHỐI THI ĐỀ XUẤT: {pred_block} (Độ tin cậy: {conf:.1f}%)")
    print(f"▶ PHÂN TÍCH NĂNG LỰC:")
    if reasons:
        for r in reasons: print(f"   - {r}")
    else:
        print(f"   - Năng lực học tập khá đồng đều, xu hướng theo sát nhóm học sinh {pred_block} của trường.")

    print(f"▶ ĐỊNH HƯỚNG NGHỀ NGHIỆP: {career_map.get(pred_block, 'Đa dạng ngành nghề.')}")
    print("▶ LỜI KHUYÊN: Bạn nên tập trung tối ưu hóa các môn thế mạnh để tăng khả năng cạnh tranh vào các trường Top đầu.")
    print("*" * 71)

# --- CHẠY THỬ NGHIỆM ---
# Giả sử một bạn điểm Sử rất cao, Anh thấp
ai_consultant_pro(7.5, 8.0, 4.5, 5.0, 5.5, 9.2)

# Giả sử một bạn điểm Toán, Anh tốt
ai_consultant_pro(8.5, 7.0, 8.8, 8.0, 6.0, 4.0)

FileNotFoundError: [Errno 2] No such file or directory: 'Goc  Thi Hoc ky1-K112-24-25 - Up.xlsx'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')